# FITS File Creation & Verification Notebook
**Project:** Astronomical Image Data Matrix Compilation  
**Objective:** Porting legacy C CFITSIO linear ramp operations to optimized Python code.

This notebook establishes an automated pipeline to construct `.fits` file types, assign appropriate headers, and visually verify dataset arrays.

### Step 1: Initialize Project Environment
Ensure your system has the proper astronomy computing environments installed.

In [ ]:
!pip install astropy numpy matplotlib

### Step 2: Import Core Modules & Configuration
Define explicit pixel matrices matching your core data dimensions ($300 \times 200$ array dimensions).

In [ ]:
import astropy.io.fits as fits
import matplotlib.pyplot as plt
import numpy as np

# Set the original structural parameters
width = 300
height = 200

### Step 3: Array Initialization & Matrix Generation
Vectorized representation of the original programmatic ramp function `array[jj][ii] = ii + jj` using NumPy index spaces.

In [ ]:
# Construct index arrays for structural height and width dimensions
jj, ii = np.indices((height, width))
data_array = (ii + jj).astype(np.int16)  # Replicates SHORT_IMG pixel structure

print(f"Data array constructed successfully with matrix shape: {data_array.shape}")

### Step 4: Construct FITS HDU & File Output
Wrap raw arrays into Primary Header Data Units (HDUs) and commit metadata headers to disk storage.

In [ ]:
# Instantiate primary header data unit wrapper
hdu = fits.PrimaryHDU(data=data_array)

# Insert explicit science metrics tracking metadata keys
hdu.header["EXPOSURE"] = (1500, "Total Exposure Time")

# Dynamically compute and inject CHECKSUM and DATASUM keys
hdu.ad

# Serialize the complete entity matrix payload to a persistent file structure
output_filename = "demo_image.fits"
hdu.writeto(output_filename, overwrite=True)

print(f"Target FITS structure written successfully to disk location: {output_filename}")

### Step 5: Readback & Quality Assurance Plotting
Re-open files from storage to read back parameters and visualize arrays.

In [ ]:
# Open the freshly minted asset interface
with fits.open("demo_image.fits") as hdul:
    print("--- COMPILED FITS METADATA DATASET ---\n")
    print(repr(hdul[0].header))
    
    # Visual confirmation plot mapping matrices
    plt.figure(figsize=(10, 6))
    plt.imshow(hdul[0].data, cmap="inferno", origin="lower")
    plt.colorbar(label="Data Value Gradient Vector")
    plt.title("FITS Output Verification Canvas (Linear Ramp Structure)")
    plt.xlabel("NAXIS1 Axis Matrix Element Range (Width)")
    plt.ylabel("NAXIS2 Axis Matrix Element Range (Height)")
    plt.show()

 ## Step 6: Create Table `nrows=1` and `nelements = 100`
Python implementation that generates the exact equivalent binary table file, including your verified checksum safety layer:

In [ ]:
import astropy.io.fits as fits
import numpy as np

# 1. Initialize sample array structures (100 elements each)
# Matching the uarray (unsigned short) and varray (unsigned int) profiles
uarray = np.arange(100, dtype=np.uint16)  # Unsigned 16-bit short integers
varray = np.arange(100, dtype=np.uint32)  # Unsigned 32-bit integers

# 2. Define the table columns
# 'format' maps to FITS column types: '100I' = 100 16-bit integers, '100J' = 100 32-bit integers
col1 = fits.Column(name="Col_1", format="100I", array=[uarray])
col2 = fits.Column(name="Col_2", format="100J", array=[varray])

# Combine the column definitions into a field layout schema
col_definitions = fits.ColDefs([col1, col2])

# 3. Create the Binary Table HDU extension layer
# This structures your extension name 'Test_table' 
table_hdu = fits.BinTableHDU.from_columns(col_definitions, name="Test_table")

# 4. Assembly: FITS tables need to append behind a structural Primary HDU skeleton
primary_hdu = fits.PrimaryHDU()
hdul = fits.HDUList([primary_hdu, table_hdu])

# 5. Apply the robust integrity validation layer to every internal component
for hdu in hdul:
    hdu.add_checksum()

# 6. Commit payload directly to file disk
output_filename = "demo_table.fits"
hdul.writeto(output_filename, overwrite=True)

print(f"Successfully generated binary table file '{output_filename}' passing strict validation!")


## Step 7: Create Large Table `nrows=99` and `nelements = 99`

### Cell 1: Imports and Configuration

In [ ]:
import astropy.io.fits as fits
import numpy as np

# Define configuration parameters
num_rows = 99
num_cols = 99
elements_per_cell = 100
output_filename = "demo_large_table.fits"


### Cell 2: Automated Table Engine (Creates 99 Rows x 99 Columns)
This cell loops dynamically to build your column definitions, bundles them into an HDU table, computes your verification checksums, and writes the file.

In [ ]:
columns_list = []

for c in range(1, num_cols + 1):
    col_name = f"Col_{c}"
    
    # Alternate data types between Unsigned Short (I) and Unsigned Int (J)
    if c % 2 != 0:
        fits_format = f"{elements_per_cell}I"  # 16-bit integers
        # Fill data matrix with the column index number for easy verification
        data_matrix = np.full((num_rows, elements_per_cell), c, dtype=np.uint16)
    else:
        fits_format = f"{elements_per_cell}J"  # 32-bit integers
        data_matrix = np.full((num_rows, elements_per_cell), c * 100, dtype=np.uint32)
        
    # Instantiate the column and add it to our list
    column = fits.Column(name=col_name, format=fits_format, array=data_matrix)
    columns_list.append(column)

# Combine columns into a definition layout
col_definitions = fits.ColDefs(columns_list)

# Build the Binary Table Extension
table_hdu = fits.BinTableHDU.from_columns(col_definitions, name="Test_table")

# Append table extension behind an empty Primary HDU structural baseline
primary_hdu = fits.PrimaryHDU()
hdul = fits.HDUList([primary_hdu, table_hdu])

# Add integrity validation checksum layers to avoid fitscheck errors
for hdu in hdul:
    hdu.add_checksum()

# Save the final file to disk
hdul.writeto(output_filename, overwrite=True)
print(f"Successfully generated '{output_filename}' with {num_rows} rows and {num_cols} columns.")

### Cell 3: Verification & Interactive Slicing Test
Run this cell to immediately load the new table, inspect its shape, check its verification tags, and pull a precise chunk of data out to confirm it works.

In [ ]:
# Open and verify the file structure 
with fits.open(output_filename) as hdul:
    table_data = hdul["Test_table"].data
    
    print("--- FILE INTEGRITY STATUS ---")
    print(f"Checksum valid: {hdul['Test_table'].verify_checksum() == 1}")
    print(f"Total Columns found: {len(table_data.columns)}")
    print(f"Total Rows found: {len(table_data)}\n")
    
    # Let's slice out an exact data point: Row 42, Column 15
    target_col = "Col_15"
    target_row = 42
    sample_cell_array = table_data[target_col][target_row]
    
    print(f"--- INTERACTIVE SLICE TEST ---")
    print(f"Data array at [{target_col}][Row {target_row}]:")
    print(f"Shape: {sample_cell_array.shape}")
    print(f"First 5 elements: {sample_cell_array[:5]} ... (all values should be 15)")
